# InstructPix2Pix club workshop

Build the model adaptation and training logic during the session. The dataset, image preprocessing, tokenization, pretrained components, and encoding helpers are provided. Work through the three **Your turn** sections in order. The completed version is in `train-implemented.ipynb`.

Run from the repository root. MagicBrush's train split is streamed; training is best on a CUDA GPU. A small batch and 256-pixel images keep the exercise more manageable.


In [ ]:
import os
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from dotenv import load_dotenv
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer

load_dotenv()
torch.manual_seed(0)


In [ ]:
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
resolution = 256  # Divisible by 8; 512 uses much more memory.
batch_size = 1
max_train_steps = 500
learning_rate = 5e-5
conditioning_dropout_prob = 0.05
output_dir = Path("checkpoints/diffusers-ip2p")

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("Training device:", device)


## Data

Each record is an original image, an edited image, and an instruction. Both images receive the same deterministic resize and normalization; instructions are tokenized for CLIP. The mask and metadata are left out of the batch.


In [ ]:
train_dataset = load_dataset(
    "osunlp/MagicBrush",
    data_files={"train": "data/train-*"},
    split="train",
    streaming=True,
    token=os.getenv("HF_TOKEN"),
)

image_transform = transforms.Compose([
    transforms.Resize((resolution, resolution), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),  # PIL RGB -> float tensor in [0, 1]
    transforms.Normalize([0.5] * 3, [0.5] * 3),  # -> [-1, 1]
])

def collate_fn(rows):
    return {
        "original_pixel_values": torch.stack([
            image_transform(row["source_img"].convert("RGB")) for row in rows
        ]),
        "edited_pixel_values": torch.stack([
            image_transform(row["target_img"].convert("RGB")) for row in rows
        ]),
        "input_ids": tokenizer(
            [row["instruction"] for row in rows],
            padding="max_length",
            truncation=True,
            max_length=tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids,
    }


## Load the Stable Diffusion components

The VAE and text encoder stay frozen. The UNet starts with four input channels and will be adapted in the first exercise. The scheduler supplies the forward noising process.


In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet")
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")
assert noise_scheduler.config.prediction_type == "epsilon"

vae.requires_grad_(False).eval()
text_encoder.requires_grad_(False).eval()
unet.to(device).train()
unet.enable_gradient_checkpointing()


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn)

# Check a real batch before the first training step.
batch = next(iter(train_loader))
assert batch["original_pixel_values"].shape == (batch_size, 3, resolution, resolution)
assert batch["edited_pixel_values"].shape == (batch_size, 3, resolution, resolution)
assert batch["input_ids"].shape == (batch_size, tokenizer.model_max_length)
for key in ("original_pixel_values", "edited_pixel_values"):
    assert batch[key].min() >= -1 and batch[key].max() <= 1
print({key: tuple(value.shape) for key, value in batch.items()})


## Your turn 1: adapt the UNet

InstructPix2Pix concatenates a four-channel noisy target latent with a four-channel source image latent. Change `unet.conv_in` to accept eight channels while retaining its pretrained behavior at initialization:

1. Copy the old convolution weights into channels 0–3 and copy its bias.
2. Initialize the new four channels to zero.
3. Update `unet.config.in_channels` so `save_pretrained` writes a loadable checkpoint.
4. Keep the new convolution on `device` and the UNet trainable.

The check below confirms that the UNet accepts eight channels and that its saved config agrees.


In [ ]:
# TODO: adapt unet.conv_in from 4 to 8 input channels.
# old_conv = unet.conv_in


In [ ]:
assert unet.conv_in.in_channels == 8
assert unet.config.in_channels == 8
print("UNet input adaptation looks ready.")


## Given: latent and text encoding helpers

The edited image is the denoising target: sample its VAE posterior and apply the VAE scaling factor. The source image is the condition: use its posterior mode **without** that factor. Text encoding returns one embedding per token. The empty prompt embedding is used when text conditioning is dropped.


In [ ]:
@torch.no_grad()
def encode_target(batch):
    pixels = batch["edited_pixel_values"].to(device)
    return vae.encode(pixels).latent_dist.sample() * vae.config.scaling_factor

@torch.no_grad()
def encode_conditions(batch):
    text_hidden = text_encoder(batch["input_ids"].to(device))[0]
    source_pixels = batch["original_pixel_values"].to(device)
    source_latents = vae.encode(source_pixels).latent_dist.mode()
    return text_hidden, source_latents

@torch.no_grad()
def encode_empty_prompt():
    empty_ids = tokenizer(
        [""], padding="max_length", truncation=True,
        max_length=tokenizer.model_max_length, return_tensors="pt",
    ).input_ids.to(device)
    return text_encoder(empty_ids)[0]

null_text_hidden = encode_empty_prompt()



## Your turn 2: write one training step

Implement `training_step(batch)` and return a scalar MSE loss. Use the helpers above to:

1. Encode the edited target, draw Gaussian noise and a random timestep for each example, then call `noise_scheduler.add_noise`.
2. Encode the instruction and source image. With probability `conditioning_dropout_prob` each, create the **text-only**, **image-only**, and **neither** conditioning cases. Replace dropped text with `null_text_hidden` and dropped image latents with zeros.
3. Concatenate the noisy target and source latents along channels, predict noise with the UNet, and compare with the sampled noise.

The UNet input should have shape `(batch_size, 8, resolution // 8, resolution // 8)`.


In [ ]:
def training_step(batch):
    # TODO: implement the InstructPix2Pix diffusion loss.
    raise NotImplementedError("Implement training_step during the workshop")


In [ ]:
# Run this check after completing training_step.
loss = training_step(batch)
assert loss.ndim == 0 and torch.isfinite(loss)
print("Initial loss:", loss.item())


## Your turn 3: write the optimization loop

Train only the UNet with `AdamW`. For each batch: clear gradients, compute the loss, backpropagate, and update the weights. Stop after `max_train_steps` batches. Save `unet.save_pretrained(output_dir / "final" / "unet")` so `demo-inference.ipynb` can load it.


In [ ]:
# TODO: create the optimizer and training loop, then save the trained UNet.
